# Fake News Detection - Fine Tuning (Baseline v1)

This is the **Baseline Version** used to achieve 85% accuracy on LIAR.
It includes:
1.  **Balanced Data** (20k Real / 12k Fake)
2.  **WELFake + LIAR + Synthetic**
3.  **No Snopes Data** (Pure Political/Synthetic Training)

## Instructions
1.  **Upload Files**: `best_model_state.bin`, `WELFake_Dataset.csv`, `train.tsv`, `valid.tsv`, `test.tsv`.
2.  **Run All**

In [ ]:
# 1. Install Dependencies
!pip install transformers torch pandas numpy matplotlib seaborn wordcloud scikit-learn

In [ ]:
# 2. Mount Drive & Define Paths
from google.colab import drive
import os

drive.mount('/content/drive')

PROJECT_PATH = '/content/drive/MyDrive/fake_news_detection'
MODEL_PATH = os.path.join(PROJECT_PATH, 'best_model_state.bin')
DATA_PATH_WELFAKE = os.path.join(PROJECT_PATH, 'WELFake_Dataset.csv')
DATA_PATH_LIAR_TRAIN = os.path.join(PROJECT_PATH, 'train.tsv')
DATA_PATH_LIAR_VALID = os.path.join(PROJECT_PATH, 'valid.tsv')
DATA_PATH_LIAR_TEST = os.path.join(PROJECT_PATH, 'test.tsv')
OUTPUT_PATH = os.path.join(PROJECT_PATH, 'best_model_state_finetuned.bin')

if not os.path.exists(PROJECT_PATH):
    os.makedirs(PROJECT_PATH, exist_ok=True)
    print(f"⚠️ Created {PROJECT_PATH}. Please upload your files there!")
else:
    print(f"✅ Project folder found at: {PROJECT_PATH}")

In [ ]:
# 3. Preprocessing Function
import re
import string

def clean_text(text):
    text = str(text).lower()
    text = re.sub('\[.*?\]', '', text)
    text = re.sub('https?://\S+|www\.\S+', '', text)
    text = re.sub('<.*?>+', '', text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub('\n', '', text)
    text = re.sub('\w*\d\w*', '', text)
    return text

In [ ]:
# 4. Load & Clean Datasets
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from wordcloud import WordCloud

# --- LIAR ---
def load_liar_manual(path):
    if not path or not os.path.exists(path):
        return pd.DataFrame(columns=['text', 'label'])
    try:
        df = pd.read_csv(path, sep='\t', header=None, usecols=[1, 2], names=['label_text', 'text'])
        label_map = {'false': 1, 'pants-fire': 1, 'barely-true': 1, 'true': 0, 'mostly-true': 0, 'half-true': 0}
        df['label'] = df['label_text'].map(label_map)
        df = df.dropna(subset=['label'])
        df['label'] = df['label'].astype(int)
        return df[['text', 'label']]
    except Exception as e:
        print(f"Error reading {path}: {e}")
        return pd.DataFrame(columns=['text', 'label'])

df_liar = pd.concat([load_liar_manual(DATA_PATH_LIAR_TRAIN), load_liar_manual(DATA_PATH_LIAR_VALID), load_liar_manual(DATA_PATH_LIAR_TEST)])

# --- WELFake ---
if os.path.exists(DATA_PATH_WELFAKE):
    print("Loading WELFake...")
    df_welfake = pd.read_csv(DATA_PATH_WELFAKE)
    df_welfake_subset = df_welfake.sample(n=20000) # Balanced Sample
else:
    df_welfake_subset = pd.DataFrame(columns=['text', 'label'])

# --- Synthetic ---
import random
def generate_synthetic_data(num_samples=1000):
    subjects = ["Aliens", "Bigfoot", "NASA", "The Government", "Scientists", "Elon Musk", "Zombies"]
    verbs = ["discovered", "revealed", "captured", "ate", "destroyed", "found", "hiding"]
    objects = ["a city on Mars", "the cure for death", "pizza", "the moon", "a portal to hell", "dinosaurs"]
    locations = ["in New York", "at Area 51", "under the ocean", "on the moon", "in a volcano"]
    data = []
    for _ in range(num_samples):
        if random.random() < 0.5:
            text = f"{random.choice(subjects)} {random.choice(verbs)} {random.choice(objects)} {random.choice(locations)}."
        else:
            text = f"BREAKING: {random.choice(subjects)} has {random.choice(verbs)} {random.choice(objects)}!"
        data.append({'text': text, 'label': 1}) # 1 = Fake
    return pd.DataFrame(data)

df_synthetic = generate_synthetic_data(1000)

# --- Combine ---
print("Combining Datasets...")
df_combined = pd.concat([df_welfake_subset[['text', 'label']], df_liar, df_synthetic])
df_combined = df_combined.sample(frac=1).reset_index(drop=True)

print(f"Total Samples: {len(df_combined)}")

In [ ]:
# 5. Comprehensive EDA

# A. Missing Values
print("Handling Missing Values...")
df_combined['text'] = df_combined['text'].fillna('')
plt.figure(figsize=(8, 4))
sns.heatmap(df_combined.isnull(), cbar=False, cmap='viridis')
plt.title('Missing Values Heatmap (Clean)')
plt.show()

# B. Class Distribution
plt.figure(figsize=(6, 4))
sns.countplot(x='label', data=df_combined)
plt.title('Class Distribution (0=Real, 1=Fake)')
plt.show()

# C. Text Length Distribution
df_combined['text_len'] = df_combined['text'].apply(lambda x: len(str(x).split()))
plt.figure(figsize=(10, 5))
sns.histplot(data=df_combined, x='text_len', hue='label', kde=True, bins=50, palette='Set2')
plt.title('Text Length Distribution (Real vs Fake)')
plt.xlabel('Number of Words')
plt.xlim(0, 1000) # Limit to 1000 words for readability
plt.show()

# D. Word Clouds
print("Generating Word Clouds...")
real_text = " ".join(df_combined[df_combined['label'] == 0]['text'].astype(str))
fake_text = " ".join(df_combined[df_combined['label'] == 1]['text'].astype(str))

plt.figure(figsize=(15, 7))
plt.subplot(1, 2, 1)
wc_real = WordCloud(width=800, height=400, background_color='white').generate(real_text)
plt.imshow(wc_real, interpolation='bilinear')
plt.title('Word Cloud - REAL News')
plt.axis('off')

plt.subplot(1, 2, 2)
wc_fake = WordCloud(width=800, height=400, background_color='black', colormap='Reds').generate(fake_text)
plt.imshow(wc_fake, interpolation='bilinear')
plt.title('Word Cloud - FAKE News')
plt.axis('off')
plt.show()

# Apply Cleaning for Training
print("\nApplying Text Cleaning...")
df_combined['text'] = df_combined['text'].apply(clean_text)

In [ ]:
# 6. Train with Validation
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import BertTokenizer, BertModel
from sklearn.model_selection import train_test_split

# Split Data
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df_combined['text'].values, df_combined['label'].values, test_size=0.1, random_state=42
)

class HybridBertBiLSTM(nn.Module):
    def __init__(self, n_classes):
        super(HybridBertBiLSTM, self).__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.lstm = nn.LSTM(input_size=768, hidden_size=128, num_layers=1, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(p=0.3)
        self.out = nn.Linear(128 * 2, n_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_state = outputs.last_hidden_state
        lstm_out, _ = self.lstm(last_hidden_state)
        out = torch.mean(lstm_out, dim=1)
        out = self.drop(out)
        return self.out(out)

class SyntheticDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]
        encoding = self.tokenizer.encode_plus(
            text, add_special_tokens=True, max_length=self.max_len,
            return_token_type_ids=False, padding='max_length',
            truncation=True, return_attention_mask=True, return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

train_dataset = SyntheticDataset(train_texts, train_labels, tokenizer)
val_dataset = SyntheticDataset(val_texts, val_labels, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

model = HybridBertBiLSTM(n_classes=2).to(DEVICE)
if os.path.exists(MODEL_PATH):
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    print("Loaded existing model weights.")

optimizer = AdamW(model.parameters(), lr=2e-5)
loss_fn = nn.CrossEntropyLoss().to(DEVICE)

history = {'loss': [], 'accuracy': []}
EPOCHS = 3

print("Starting Training...")
model.train()
for epoch in range(EPOCHS):
    epoch_loss = 0
    correct_predictions = 0
    total_examples = 0
    
    for i, d in enumerate(train_loader):
        input_ids = d['input_ids'].to(DEVICE)
        attention_mask = d['attention_mask'].to(DEVICE)
        targets = d['labels'].to(DEVICE)
        
        outputs = model(input_ids, attention_mask)
        loss = loss_fn(outputs, targets)
        
        _, preds = torch.max(outputs, dim=1)
        correct_predictions += torch.sum(preds == targets)
        total_examples += targets.size(0)
        
        history['loss'].append(loss.item())
        epoch_loss += loss.item()

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        optimizer.zero_grad()
        
        if i % 50 == 0:
            print(f"Epoch {epoch+1} Batch {i} Loss: {loss.item():.4f}")
            
    epoch_acc = correct_predictions.double() / total_examples
    history['accuracy'].append(epoch_acc.item())
    print(f"Epoch {epoch+1} Avg Loss: {epoch_loss/len(train_loader):.4f} | Accuracy: {epoch_acc:.4f}")

torch.save(model.state_dict(), OUTPUT_PATH)
print(f"Saved model to {OUTPUT_PATH}")

In [ ]:
# 7. Final Visualizations
from sklearn.metrics import confusion_matrix, classification_report

# A. Training Curves
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history['loss'], label='Training Loss', color='red')
plt.title('Training Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history['accuracy'], label='Training Accuracy', color='green', marker='o')
plt.title('Training Accuracy')
plt.legend()
plt.show()

# B. Confusion Matrix (Validation)
print("Running Validation...")
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for d in val_loader:
        input_ids = d['input_ids'].to(DEVICE)
        attention_mask = d['attention_mask'].to(DEVICE)
        targets = d['labels'].to(DEVICE)
        
        outputs = model(input_ids, attention_mask)
        _, preds = torch.max(outputs, dim=1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(targets.cpu().numpy())

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
plt.title('Confusion Matrix (Validation Set)')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=['Real', 'Fake']))